# Revision runs (everything that needs the Mac)

Runs the experiments added after the review, using the repository's own
command-line tool, so the results come from exactly the code on GitHub:

1. ISO-NE load, **monthly refits over 2025** (13 rolling windows): naive, trees, deep models, Chronos
2. ISO-NE load with **real archived weather forecasts** at test time
3. ISO-NE price with the **bidding-gate fix** (all models incl. Chronos)
4. Tetouan with the **equal-information features** (deep models and Chronos)
5. EPF markets: robust significance tests and comparison with the **published Lago et al. forecasts**

Every step saves its results and is skipped next time if already done, so you
can stop and restart the notebook at any time. At the end, send
`~/Downloads/revision_results.zip`.

In [7]:
# ===================================================================
# STEP 1 — SETTINGS
# ===================================================================
import os, sys, subprocess, shutil, time, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

HOME = Path.home()
REPO = HOME / "Downloads" / "open-forecast"            # your GitHub clone (updated to v5)
OLD_DATA = HOME / "Downloads" / "open-forecasting-framework-v2" / "data" / "raw"
OLD_RESULTS = HOME / "Downloads" / "open-forecast-complete" / "results"   # earlier full run (EPF forecasts)
EPF_DIRS = [HOME / "Downloads" / "open-forecast-complete" / "data" / "raw" / "epf", HOME / "Downloads"]
DEVICE = "mps"                 # "cpu" if you get MPS errors

RUN_DEEP = True                # deep models in the rolling evaluation (the longest step)
RUN_FORECAST_WEATHER = True    # needs internet (Open-Meteo)
RUN_SEEDS = False              # extra: LSTM with 5 seeds in the rolling evaluation (~3 h)
FULL_DEEP = False              # True: Transformer models in all three sets (several extra hours)

os.environ["OPEN_FORECAST_DEVICE"] = DEVICE
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.chdir(REPO)
RES = REPO / "results"
MODES = ["weather_time", "weather_time_history", "weather_time_history_weekly"]
print("Repository:", REPO, "| exists:", REPO.exists())
print("Version check (should say v5):", (REPO / "CHANGELOG.md").read_text().splitlines()[2])


def sh(*args, log=None):
    """Run a command, stream its output here and stop on error."""
    print("$", " ".join(map(str, args)), flush=True)
    start = time.time()
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not line.lstrip().startswith(("lear: recalibrated", "hist_gradient_boosting: recal", "extra_trees: recal")):
            print(line, end="", flush=True)
    proc.wait()
    if log:
        Path(log).write_text("".join(lines))
    print(f"[{(time.time() - start) / 60:.1f} min, exit code {proc.returncode}]")
    if proc.returncode != 0:
        raise RuntimeError("command failed; see output above")


def cli(*args):
    sh(sys.executable, "-u", "-W", "ignore", "-m", "open_forecast.cli", *args)

Repository: /Users/SUNYLoaner/Downloads/open-forecast | exists: True
Version check (should say v5): ## v5 (November 2026): response to an external review


In [8]:
# ===================================================================
# STEP 2 — INSTALL THE PACKAGE AND PUT THE DATA WHERE THE CONFIGS EXPECT IT
# ===================================================================
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".[all]")

wanted = {REPO / "data/raw/isone" / f"{y}_smd_hourly.xlsx": OLD_DATA / "isone" / f"{y}_smd_hourly.xlsx" for y in (2023, 2024, 2025)}
wanted[REPO / "data/raw/power_consumption.csv"] = OLD_DATA / "power_consumption.csv"
for target, source in wanted.items():
    if not target.exists() and source.exists():
        target.parent.mkdir(parents=True, exist_ok=True); shutil.copy(source, target)
    print("OK     " if target.exists() else "MISSING", target)
for market in ["PJM", "NP", "DE", "BE", "FR"]:
    target = REPO / "data/raw/epf" / f"{market}.csv"
    for d in EPF_DIRS:
        if not target.exists() and (d / f"{market}.csv").exists():
            target.parent.mkdir(parents=True, exist_ok=True); shutil.copy(d / f"{market}.csv", target)
    print("OK     " if target.exists() else "MISSING (only needed for step 8)", target)

$ /opt/miniconda3/bin/python -m pip install -q -e .[all]
[0.0 min, exit code 0]
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/isone/2023_smd_hourly.xlsx
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/isone/2024_smd_hourly.xlsx
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/isone/2025_smd_hourly.xlsx
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/power_consumption.csv
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/epf/PJM.csv
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/epf/NP.csv
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/epf/DE.csv
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/epf/BE.csv
OK      /Users/SUNYLoaner/Downloads/open-forecast/data/raw/epf/FR.csv


In [9]:
# ===================================================================
# STEP 3 — TESTS (should all pass, including the new leakage audit)
# ===================================================================
sh(sys.executable, "-m", "pytest", "-q")

$ /opt/miniconda3/bin/python -m pytest -q
..................................                                       [100%]
34 passed in 5.01s
[0.1 min, exit code 0]


In [10]:
# ===================================================================
# STEP 4 — ISO-NE LOAD, MONTHLY REFITS: NAIVE + TREES (13 windows)
# ===================================================================
cli("rolling", "--config", "configs/isone_rolling.yaml",
    "--models", "persistence", "seasonal_naive", "random_forest", "extra_trees", "hist_gradient_boosting",
    "--modes", *MODES, "--output", RES / "isone_rolling_trees")
display(pd.read_csv(RES / "isone_rolling_trees" / "rolling_summary.csv").round(2))

$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli rolling --config configs/isone_rolling.yaml --models persistence seasonal_naive random_forest extra_trees hist_gradient_boosting --modes weather_time weather_time_history weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/isone_rolling_trees
fold 0 (2025-01-01) done
fold 1 (2025-01-29) done
fold 2 (2025-02-26) done
fold 3 (2025-03-26) done
fold 4 (2025-04-23) done
fold 5 (2025-05-21) done
fold 6 (2025-06-18) done
fold 7 (2025-07-16) done
fold 8 (2025-08-13) done
fold 9 (2025-09-10) done
fold 10 (2025-10-08) done
fold 11 (2025-11-05) done
fold 12 (2025-12-03) done
                 model                feature_mode  folds  mean_rmse  median_rmse  mean_rank  wins
hist_gradient_boosting weather_time_history_weekly     13  94.241729    91.390116   1.538462     6
hist_gradient_boosting        weather_time_history     13  94.087556    88.537217   1.615385     6
           extra_trees        weat

,model,feature_mode,folds,mean_rmse,median_rmse,mean_rank,wins
0,hist_gradient_boosting,weather_time_history_weekly,13,94.24,91.39,1.54,6
1,hist_gradient_boosting,weather_time_history,13,94.09,88.54,1.62,6
2,extra_trees,weather_time_history,13,108.67,98.68,4.62,0
3,hist_gradient_boosting,weather_time,13,106.59,104.96,4.69,1
4,extra_trees,weather_time_history_weekly,13,109.76,101.53,4.92,0
5,extra_trees,weather_time,13,108.71,105.18,5.77,0
6,random_forest,weather_time_history,13,113.11,104.67,6.85,0
7,random_forest,weather_time_history_weekly,13,115.19,108.62,7.46,0
8,random_forest,weather_time,13,113.88,113.63,7.54,0
9,persistence,weather_time_history,13,195.62,173.98,10.08,0


In [11]:
# ===================================================================
# STEP 5 — ISO-NE LOAD, MONTHLY REFITS: CHRONOS (zero-shot, scored on the same windows)
# ===================================================================
cli("rolling", "--config", "configs/isone_rolling.yaml",
    "--models", "chronos_bolt_small", "chronos2",
    "--modes", "weather_time_history_weekly", "--output", RES / "isone_rolling_foundation")
display(pd.read_csv(RES / "isone_rolling_foundation" / "rolling_summary.csv").round(2))

$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli rolling --config configs/isone_rolling.yaml --models chronos_bolt_small chronos2 --modes weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/isone_rolling_foundation
fold 0 (2025-01-01) done
fold 1 (2025-01-29) done
fold 2 (2025-02-26) done
fold 3 (2025-03-26) done
fold 4 (2025-04-23) done
fold 5 (2025-05-21) done
fold 6 (2025-06-18) done
fold 7 (2025-07-16) done
fold 8 (2025-08-13) done
fold 9 (2025-09-10) done
fold 10 (2025-10-08) done
fold 11 (2025-11-05) done
fold 12 (2025-12-03) done
             model                feature_mode  folds  mean_rmse  median_rmse  mean_rank  wins
          chronos2 weather_time_history_weekly     13 105.582193    97.922060        1.0    13
chronos_bolt_small weather_time_history_weekly     13 173.211962   143.949428        2.0     0
[0.0 min, exit code 0]


,model,feature_mode,folds,mean_rmse,median_rmse,mean_rank,wins
0,chronos2,weather_time_history_weekly,13,105.58,97.92,1.0,13
1,chronos_bolt_small,weather_time_history_weekly,13,173.21,143.95,2.0,0


In [12]:
# ===================================================================
# STEP 6 — ISO-NE PRICE WITH THE BIDDING-GATE FIX (all models, weekly refits)
# ===================================================================
cli("price", "--config", "configs/isone_price.yaml", "--output", RES / "price_isone_price",
    "--models", "naive", "lear", "hist_gradient_boosting", "extra_trees",
    "chronos_bolt_small", "chronos_bolt_base", "chronos2")
display(pd.read_csv(RES / "price_isone_price" / "price_metrics.csv").round(3))

$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli price --config configs/isone_price.yaml --output /Users/SUNYLoaner/Downloads/open-forecast/results/price_isone_price --models naive lear hist_gradient_boosting extra_trees chronos_bolt_small chronos_bolt_base chronos2
{'model': 'naive', 'information_set': '-', 'test_days': 365, 'mae': 19.844449771689497, 'rmse': 34.460806595796285, 'smape_pct': 26.40118536200429, 'rmae': 1.0, 'seconds': 0.0}
  lear|price_only: loaded cached forecasts
{'model': 'lear', 'information_set': 'price_only', 'test_days': 365, 'mae': 13.158076317357438, 'rmse': 24.04987773662203, 'smape_pct': 17.779755063865455, 'rmae': 0.6630607786429544, 'seconds': 0.0}
  hist_gradient_boosting|price_only: loaded cached forecasts
{'model': 'hist_gradient_boosting', 'information_set': 'price_only', 'test_days': 365, 'mae': 14.37791047644534, 'rmse': 25.775970178115767, 'smape_pct': 19.228244182562694, 'rmae': 0.7245305685903756, 'seconds': 0.0}
  extra_trees|price_

,model,information_set,test_days,mae,rmse,smape_pct,rmae,seconds
0,naive,-,365,19.844,34.461,26.401,1.000,0.0
1,lear,price_only,365,13.158,24.050,17.780,0.663,0.0
2,hist_gradient_boosting,price_only,365,14.378,25.776,19.228,0.725,0.0
3,extra_trees,price_only,365,13.873,25.673,18.655,0.699,0.0
4,chronos_bolt_small,price_only,365,13.101,23.696,17.906,0.660,0.0
5,chronos_bolt_base,price_only,365,13.198,24.132,17.863,0.665,0.0
6,chronos2,price_only,365,12.398,22.629,16.994,0.625,0.0
7,lear,price_exogenous,365,12.448,23.176,16.800,0.627,0.0
8,hist_gradient_boosting,price_exogenous,365,12.512,22.317,16.862,0.631,20122.8
9,extra_trees,price_exogenous,365,12.885,24.174,17.316,0.649,0.0


In [13]:
# ===================================================================
# STEP 7 — TETOUAN: DEEP MODELS AND CHRONOS WITH THE NEW INFORMATION SETS
# ===================================================================
cli("run", "--config", "configs/tetouan_v2.yaml", "--models", "persistence", "seasonal_naive",
    "chronos_bolt_small", "chronos_bolt_base", "chronos2", "--modes", *MODES,
    "--output", RES / "tetouan_v2_foundation")
if RUN_DEEP:
    cli("run", "--config", "configs/tetouan_v2.yaml", "--models", "persistence", "seasonal_naive",
        "lstm", "cross_attention_transformer", "patchtst", "--modes", *MODES, "--output", RES / "tetouan_v2_deep")
cli("rolling", "--config", "configs/tetouan_rolling.yaml", "--models", "chronos_bolt_small", "chronos2",
    "--modes", "weather_time_history_weekly", "--output", RES / "tetouan_rolling_foundation")
if RUN_DEEP:
    cli("rolling", "--config", "configs/tetouan_rolling.yaml", "--models", "lstm", "cross_attention_transformer",
        "patchtst", "--modes", *MODES, "--output", RES / "tetouan_rolling_deep")

$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli run --config configs/tetouan_v2.yaml --models persistence seasonal_naive chronos_bolt_small chronos_bolt_base chronos2 --modes weather_time weather_time_history weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/tetouan_v2_foundation
W1111 13:21:31.842000 42911 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

Loading weights: 100%|██████████| 143/143 [00:00<00:00, 50491.24it/s]

Loading weights: 100%|██████████| 269/269 [00:00<00:00, 1526.94it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 1194.11it/s]
             model                feature_mode split     scope        rmse         mae  nrmse_pct       r2
       persistence        weather_time_history  test aggregate 1584.494444 1029.490190   6.027053 0.923846
chronos_bolt_small weather_time_history_weekly  test aggregate  971.794123  672.818669 

In [14]:
# ===================================================================
# STEP 8 — EPF MARKETS: ROBUST TESTS + PUBLISHED LAGO ET AL. FORECASTS
# Uses the EPF forecasts you already computed (no re-run). The published
# DNN / LEAR ensemble forecasts are downloaded from the epftoolbox GitHub repo.
# ===================================================================
import numpy as np, urllib.request
from open_forecast.significance import multivariate_dm, holm

PUB = REPO / "data/external/epftoolbox_forecasts"; PUB.mkdir(parents=True, exist_ok=True)
rows = []
for market in ["PJM", "NP", "DE", "BE", "FR"]:
    pub_file = PUB / f"Forecasts_{market}_DNN_LEAR_ensembles.csv"
    if not pub_file.exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/jeslago/epftoolbox/master/forecasts/"
                                   f"Forecasts_{market}_DNN_LEAR_ensembles.csv", pub_file)
    pub = pd.read_csv(pub_file, index_col=0, parse_dates=True)
    ours_dir = next(d for d in [RES / f"price_epf_{market.lower()}", OLD_RESULTS / f"price_epf_{market.lower()}"]
                    if (d / "price_predictions.csv").exists() or (d / "price_predictions.csv.gz").exists())
    f = ours_dir / "price_predictions.csv"
    ours = pd.read_csv(f if f.exists() else ours_dir / "price_predictions.csv.gz", parse_dates=["day"])
    ours["hour"] = ours["hour"].astype(str).str.lstrip("h").astype(int)      # "h7" -> 7
    ours["key"] = ours["model"] + "|" + ours["information_set"].fillna("-")
    wide = {k: g.pivot(index="day", columns="hour", values="predicted") for k, g in ours.groupby("key")}
    truth = ours.drop_duplicates(["day", "hour"]).pivot(index="day", columns="hour", values="actual")
    pubd = {c: pub[c].groupby(pub.index.normalize()).apply(lambda s: s.to_numpy()).apply(pd.Series) for c in
            ["LEAR Ensemble", "DNN Ensemble"]}
    days = truth.index.intersection(pubd["DNN Ensemble"].index)
    naive_key = next(k for k in wide if k.startswith("naive|"))
    naive_mae = np.abs(wide[naive_key].loc[days].to_numpy() - truth.loc[days].to_numpy()).mean()
    candidates = {"Chronos-2 + exogenous": wide["chronos2|price_exogenous"], "LEAR (ours) + exogenous": wide["lear|price_exogenous"],
                  "LEAR ensemble (published)": pubd["LEAR Ensemble"], "DNN ensemble (published)": pubd["DNN Ensemble"]}
    t = truth.loc[days].to_numpy()
    for name, frame in candidates.items():
        p = frame.loc[days].to_numpy()
        row = {"market": market, "model": name, "days": len(days), "rMAE": np.abs(p - t).mean() / naive_mae}
        c2 = candidates["Chronos-2 + exogenous"].loc[days].to_numpy()
        if name != "Chronos-2 + exogenous":   # H1: Chronos-2 more accurate than this model
            row["p_iid"] = multivariate_dm(t, p, c2)[1]
            row["p_newey_west"] = multivariate_dm(t, p, c2, lags=7)[1]
            row["p_other_better_nw"] = multivariate_dm(t, c2, p, lags=7)[1]
        rows.append(row)
epf = pd.DataFrame(rows)
mask = epf["p_newey_west"].notna()
epf.loc[mask, "p_holm_nw"] = holm(epf.loc[mask, "p_newey_west"])
(RES / "epf_published_comparison").mkdir(exist_ok=True)
epf.to_csv(RES / "epf_published_comparison" / "comparison.csv", index=False)
display(epf.round(4))

,market,model,days,rMAE,p_iid,p_newey_west,p_other_better_nw,p_holm_nw
0,PJM,Chronos-2 + exogenous,728,0.6135,NaN,NaN,NaN,NaN
1,PJM,LEAR (ours) + exogenous,728,0.6359,0.0187,0.0185,0.9815,0.0741
2,PJM,LEAR ensemble (published),728,0.6218,0.1988,0.2149,0.7851,0.4298
3,PJM,DNN ensemble (published),728,0.5907,0.9814,0.9851,0.0149,0.9851
4,NP,Chronos-2 + exogenous,728,0.4876,NaN,NaN,NaN,NaN
5,NP,LEAR (ours) + exogenous,728,0.6407,0.0000,0.0000,1.0000,0.0000
6,NP,LEAR ensemble (published),728,0.5491,0.0000,0.0000,1.0000,0.0000
7,NP,DNN ensemble (published),728,0.5319,0.0000,0.0000,1.0000,0.0000
8,DE,Chronos-2 + exogenous,728,0.3945,NaN,NaN,NaN,NaN
9,DE,LEAR (ours) + exogenous,728,0.4939,0.0000,0.0000,1.0000,0.0000


In [15]:
# ===================================================================
# STEP 9 — FORECAST COMBINATION: CHRONOS-2 + LEAR (equal-weight average)
# No new training: averages the two forecasts already computed, then tests
# the combination against every model and the published Lago et al. ensembles.
# ===================================================================
ens_rows = []
targets = [("ISO-NE", RES / "price_isone_price", None)]
for market in ["PJM", "NP", "DE", "BE", "FR"]:
    dest = RES / f"price_epf_{market.lower()}"
    dest.mkdir(parents=True, exist_ok=True)
    if not (dest / "price_predictions.csv.gz").exists():
        for src in [dest / "price_predictions.csv", OLD_RESULTS / f"price_epf_{market.lower()}" / "price_predictions.csv"]:
            if src.exists():
                pd.read_csv(src).to_csv(dest / "price_predictions.csv.gz", index=False, compression="gzip")
                break
    targets.append((market, dest, PUB / f"Forecasts_{market}_DNN_LEAR_ensembles.csv"))
for market, folder, pub_file in targets:
    args = [sys.executable, "analysis/price_ensemble.py", folder] + ([pub_file] if pub_file else [])
    sh(*args)
    m = pd.read_csv(folder / "ensemble_metrics.csv")
    ens_rows.append(m.assign(market=market))
ens = pd.concat(ens_rows)
display(ens.pivot_table(index="forecast", columns="market", values="rmae").round(3))


$ /opt/miniconda3/bin/python analysis/price_ensemble.py /Users/SUNYLoaner/Downloads/open-forecast/results/price_isone_price
                                  forecast  days     mae    rmse   rmae
                  chronos2|price_exogenous   365 10.2791 18.5487 0.5180
ensemble: Chronos-2 + LEAR (equal weights)   365 10.5351 19.5785 0.5309
                       chronos2|price_only   365 12.3983 22.6293 0.6248
                      lear|price_exogenous   365 12.4478 23.1760 0.6273
    hist_gradient_boosting|price_exogenous   365 12.5124 22.3173 0.6305
               extra_trees|price_exogenous   365 12.8852 24.1743 0.6493
             chronos_bolt_small|price_only   365 13.1007 23.6959 0.6602
                           lear|price_only   365 13.1581 24.0499 0.6631
              chronos_bolt_base|price_only   365 13.1984 24.1320 0.6651
                    extra_trees|price_only   365 13.8734 25.6734 0.6991
         hist_gradient_boosting|price_only   365 14.3779 25.7760 0.7245
            

market,BE,DE,FR,ISO-NE,NP,PJM
forecast,,,,,,
DNN ensemble (published),0.716,0.425,0.649,NaN,0.532,0.591
LEAR ensemble (published),0.748,0.449,0.668,NaN,0.549,0.622
chronos2|price_exogenous,0.704,0.394,0.622,0.518,0.488,0.614
chronos2|price_only,0.771,0.645,0.716,0.625,0.612,0.711
chronos_bolt_base|price_only,0.772,0.672,0.740,0.665,0.643,0.745
chronos_bolt_small|price_only,0.791,0.670,0.739,0.660,0.657,0.745
ensemble: Chronos-2 + LEAR (equal weights),0.705,0.400,0.622,0.531,0.519,0.588
extra_trees|price_exogenous,0.818,0.584,0.811,0.649,0.779,0.664
extra_trees|price_only,0.811,0.696,0.792,0.699,0.776,0.738


In [17]:
# ===================================================================
# STEP 10 — ISO-NE LOAD WITH REAL WEATHER FORECASTS AT TEST TIME
# Forecasts issued >= 48 h before the valid hour (Open-Meteo, NCEP GFS).
# Models are trained on observed weather, as an operator would.
# ===================================================================
if RUN_FORECAST_WEATHER:
    fc = REPO / "data/raw/isone/weather_forecasts_previous_day2.csv"
    if not fc.exists():
        sh(sys.executable, "scripts/fetch_weather_forecasts.py", "--lead", "2")
    # check: the forecasts must track the observed weather closely
    import open_forecast.isone as iso, yaml
    cfg = yaml.safe_load(open("configs/isone_rolling.yaml"))
    hourly, _ = iso.load_isone_smd(cfg["dataset"]["paths"], cfg["dataset"]["target_columns"])
    forecast = pd.read_csv(fc, parse_dates=["ts"]).set_index("ts")
    joined = hourly[forecast.columns].join(forecast, rsuffix="_fc").loc["2025"]
    check = pd.DataFrame({c: {"corr": joined[c].corr(joined[c + "_fc"]),
                              "MAE_F": (joined[c] - joined[c + "_fc"]).abs().mean()} for c in forecast.columns}).T
    display(check.round(3))
    cli("rolling", "--config", "configs/isone_rolling.yaml", "--weather-forecast", fc,
        "--models", "hist_gradient_boosting", "random_forest", "chronos2", "persistence",
        "--modes", *MODES, "--output", RES / "isone_rolling_forecastweather_trees")
    if RUN_DEEP:
        cli("rolling", "--config", "configs/isone_rolling.yaml", "--weather-forecast", fc,
            "--models", "lstm", "--modes", *MODES, "--output", RES / "isone_rolling_forecastweather_lstm")

$ /opt/miniconda3/bin/python scripts/fetch_weather_forecasts.py --lead 2
ME: 100.0% of hours available
NH: 100.0% of hours available
VT: 100.0% of hours available
CT: 100.0% of hours available
RI: 100.0% of hours available
SEMA: 100.0% of hours available
WCMA: 100.0% of hours available
NEMA: 100.0% of hours available
wrote data/raw/isone/weather_forecasts_previous_day2.csv: 9504 hours, 100.0% complete
[2.9 min, exit code 0]


,corr,MAE_F
ME_Dry_Bulb,0.977,3.082
ME_Dew_Point,0.971,3.805
NH_Dry_Bulb,0.973,3.779
NH_Dew_Point,0.972,3.753
VT_Dry_Bulb,0.981,4.323
VT_Dew_Point,0.972,3.581
CT_Dry_Bulb,0.975,3.300
CT_Dew_Point,0.972,3.482
RI_Dry_Bulb,0.981,2.643
RI_Dew_Point,0.975,3.286


$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli rolling --config configs/isone_rolling.yaml --weather-forecast /Users/SUNYLoaner/Downloads/open-forecast/data/raw/isone/weather_forecasts_previous_day2.csv --models hist_gradient_boosting random_forest chronos2 persistence --modes weather_time weather_time_history weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/isone_rolling_forecastweather_trees
W1111 23:33:50.614000 47178 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 66663.40it/s]
fold 0 (2025-01-01) done
fold 1 (2025-01-29) done
fold 2 (2025-02-26) done
fold 3 (2025-03-26) done
fold 4 (2025-04-23) done
fold 5 (2025-05-21) done
fold 6 (2025-06-18) done
fold 7 (2025-07-16) done
fold 8 (2025-08-13) done
fold 9 (2025-09-10) done
fold 10 (2025-10-08) done
fold 11 (2025-11-05) done
fold 12 (2025-12-03) 

In [18]:
# ===================================================================
# STEP 11 — ISO-NE LOAD, MONTHLY REFITS: DEEP MODELS (longest step; resumable)
# ===================================================================
if RUN_DEEP:
    for model in ["lstm", "cross_attention_transformer", "patchtst"]:
        modes = MODES if (model == "lstm" or FULL_DEEP) else ["weather_time_history_weekly"]
        cli("rolling", "--config", "configs/isone_rolling.yaml", "--models", model,
            "--modes", *modes, "--output", RES / f"isone_rolling_{model}")
        display(pd.read_csv(RES / f"isone_rolling_{model}" / "rolling_summary.csv").round(2))
if RUN_SEEDS:
    cli("rolling", "--config", "configs/isone_rolling.yaml", "--models", "lstm",
        "--modes", "weather_time_history_weekly", "--seeds", "1", "2", "3", "4",
        "--output", RES / "isone_rolling_lstm_seeds")

$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli rolling --config configs/isone_rolling.yaml --models lstm --modes weather_time weather_time_history weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/isone_rolling_lstm
fold 0 (2025-01-01) done
fold 1 (2025-01-29) done
fold 2 (2025-02-26) done
fold 3 (2025-03-26) done
fold 4 (2025-04-23) done
fold 5 (2025-05-21) done
fold 6 (2025-06-18) done
fold 7 (2025-07-16) done
fold 8 (2025-08-13) done
fold 9 (2025-09-10) done
fold 10 (2025-10-08) done
fold 11 (2025-11-05) done
fold 12 (2025-12-03) done
model                feature_mode  folds  mean_rmse  median_rmse  mean_rank  wins
 lstm        weather_time_history     13  88.895662    87.799657   1.538462     7
 lstm weather_time_history_weekly     13  87.920558    83.156790   1.538462     6
 lstm                weather_time     13 102.409662    99.886410   2.923077     0
[441.3 min, exit code 0]


,model,feature_mode,folds,mean_rmse,median_rmse,mean_rank,wins
0,lstm,weather_time_history,13,88.90,87.80,1.54,7
1,lstm,weather_time_history_weekly,13,87.92,83.16,1.54,6
2,lstm,weather_time,13,102.41,99.89,2.92,0


$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli rolling --config configs/isone_rolling.yaml --models cross_attention_transformer --modes weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/isone_rolling_cross_attention_transformer
fold 0 (2025-01-01) done
fold 1 (2025-01-29) done
fold 2 (2025-02-26) done
fold 3 (2025-03-26) done
fold 4 (2025-04-23) done
fold 5 (2025-05-21) done
fold 6 (2025-06-18) done
fold 7 (2025-07-16) done
fold 8 (2025-08-13) done
fold 9 (2025-09-10) done
fold 10 (2025-10-08) done
fold 11 (2025-11-05) done
fold 12 (2025-12-03) done
                      model                feature_mode  folds  mean_rmse  median_rmse  mean_rank  wins
cross_attention_transformer weather_time_history_weekly     13  98.041654    98.185073        1.0    13
[424.4 min, exit code 0]


,model,feature_mode,folds,mean_rmse,median_rmse,mean_rank,wins
0,cross_attention_transformer,weather_time_history_weekly,13,98.04,98.19,1.0,13


$ /opt/miniconda3/bin/python -u -W ignore -m open_forecast.cli rolling --config configs/isone_rolling.yaml --models patchtst --modes weather_time_history_weekly --output /Users/SUNYLoaner/Downloads/open-forecast/results/isone_rolling_patchtst
fold 0 (2025-01-01) done
fold 1 (2025-01-29) done
fold 2 (2025-02-26) done
fold 3 (2025-03-26) done
fold 4 (2025-04-23) done
fold 5 (2025-05-21) done
fold 6 (2025-06-18) done
fold 7 (2025-07-16) done
fold 8 (2025-08-13) done
fold 9 (2025-09-10) done
fold 10 (2025-10-08) done
fold 11 (2025-11-05) done
fold 12 (2025-12-03) done
   model                feature_mode  folds  mean_rmse  median_rmse  mean_rank  wins
patchtst weather_time_history_weekly     13 102.386554     96.78576        1.0    13
[384.0 min, exit code 0]


,model,feature_mode,folds,mean_rmse,median_rmse,mean_rank,wins
0,patchtst,weather_time_history_weekly,13,102.39,96.79,1.0,13


In [19]:
# ===================================================================
# STEP 12 — SIGNIFICANCE TESTS AND PAPER TABLES
# ===================================================================
for d in sorted(RES.glob("*rolling*")):
    f = d / "rolling_test_predictions.csv.gz"
    print(d.name, "predictions:", f.exists())
sh(sys.executable, "analysis/paper_tables.py", "results")

isone_rolling_cross_attention_transformer predictions: True
isone_rolling_forecastweather_lstm predictions: True
isone_rolling_forecastweather_trees predictions: True
isone_rolling_foundation predictions: True
isone_rolling_lstm predictions: True
isone_rolling_patchtst predictions: True
isone_rolling_trees predictions: True
tetouan_rolling_deep predictions: True
tetouan_rolling_foundation predictions: True
tetouan_rolling_trees predictions: True
$ /opt/miniconda3/bin/python analysis/paper_tables.py results

ISO-NE load, 13 monthly-refit windows (MW)
                          model set  origins    rmse     mae  nrmse_pct  mean_zonal_nrmse_pct  mape_pct  peak_hour_rmse  mean_rank  wins
0                          lstm  B+     8736   89.82   59.84       5.44                  5.66      4.36          130.87       2.23     5
1                          lstm   B     8736   90.24   60.35       5.47                  5.73      4.45          126.24       2.46     6
2        hist_gradient_boosting  

In [20]:
# ===================================================================
# STEP 13 — ZIP THE RESULTS (no raw data, no model weights)
# ===================================================================
zip_path = HOME / "Downloads" / "revision_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RES.rglob("*"):
        in_fold = any(part.startswith("fold_") for part in f.parts)
        if f.is_file() and f.suffix in (".csv", ".gz", ".json", ".txt", ".png", ".pdf") \
                and f.name not in ("predictions.csv", "price_predictions.csv") and "forecasts" not in f.parts \
                and not (in_fold and f.suffix == ".gz"):
            z.write(f, f.relative_to(REPO))
print("Results archive:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")

Results archive: /Users/SUNYLoaner/Downloads/revision_results.zip (45.5 MB)
